# 8장 · 표본으로 전체 알기 — 추정과 신뢰구간 📊

웹 챕터 **"표본으로 전체 알기 — 추정과 신뢰구간"** 과 짝을 이루는 Python 실습입니다.
신뢰구간 100개를 직접 만들어 "95% 신뢰"의 뜻을 확인하고, 여론조사의 ±3.1%p 를 계산합니다.

준비물: `pip install -r requirements.txt`


> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)


In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch8')

## 0. 준비: 1.96 은 어디서 오나

scipy 의 정규분포 도구로 "가운데 95% 를 담는 z 값"을 직접 꺼내 봅니다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(42)

# 표준정규분포에서 아래쪽 97.5% 지점 = 가운데 95% 를 담는 경계
z90 = stats.norm.ppf(0.95)
z95 = stats.norm.ppf(0.975)
z99 = stats.norm.ppf(0.995)
print(f'신뢰수준 90%  ->  z = {z90:.3f}')
print(f'신뢰수준 95%  ->  z = {z95:.3f}   (뉴스의 1.96)')
print(f'신뢰수준 99%  ->  z = {z99:.3f}')

## 1. 신뢰구간 하나 만들어 보기

참값을 아는 '신의 시점'으로 실험합니다: 모집단은 μ=50, σ=10.
표본 25개를 뽑아 95% 신뢰구간을 만들고 참값이 들어있는지 봅니다.


In [ ]:
MU, SIGMA, n = 50, 10, 25

sample = rng.normal(MU, SIGMA, size=n)   # 표본 25개
xbar = sample.mean()                     # 표본평균 (통계량)
se = SIGMA / np.sqrt(n)                  # 표준오차 (7장)
lo, hi = xbar - 1.96 * se, xbar + 1.96 * se

print(f'표본평균     = {xbar:.2f}')
print(f'표준오차 SE  = {se:.2f}')
print(f'95% 신뢰구간 = [{lo:.2f}, {hi:.2f}]')
print(f'참값 μ=50 포함? -> {lo <= MU <= hi}')

## 2. 신뢰구간 100개: "95%" 의 진짜 뜻

같은 조사를 100번 반복해 구간 100개를 만들면, 약 95개는 참값을 포함하고
**약 5개는 놓칩니다**. 놓친 구간을 빨간색으로 표시해 봅니다.


In [ ]:
k = 100
samples = rng.normal(MU, SIGMA, size=(k, n))
xbars = samples.mean(axis=1)
half = 1.96 * SIGMA / np.sqrt(n)
hit = (xbars - half <= MU) & (MU <= xbars + half)

plt.figure(figsize=(7, 10))
for i, (m, h) in enumerate(zip(xbars, hit)):
    color = 'silver' if h else 'crimson'
    plt.plot([m - half, m + half], [i, i], color=color, lw=1.5)
    plt.plot(m, i, 'o', color=color, ms=2.5)
plt.axvline(MU, color='navy', ls='--', label='참값 μ = 50')
plt.title(f'95% 신뢰구간 100개 · 놓친 구간(빨강): {k - hit.sum()}개')
plt.xlabel('값')
plt.ylabel('실험 번호')
plt.legend()
plt.show()

print(f'참값 포함: {hit.sum()} / {k}개')

## 3. n 이 커지면 구간이 좁아진다

정밀도(구간 폭)는 √n 을 따라갑니다. 오차를 절반으로 줄이려면 표본이 4배 필요해요.


In [ ]:
ns = np.arange(10, 401, 5)
widths = 2 * 1.96 * SIGMA / np.sqrt(ns)   # 구간 전체 폭

plt.figure(figsize=(8, 3.5))
plt.plot(ns, widths)
plt.xlabel('표본 크기 n')
plt.ylabel('구간 전체 폭')
plt.title('n 이 커질수록 구간은 좁아진다 (√n 이라 점점 완만하게)')
plt.show()

for n_ in [25, 100, 400]:
    print(f'n={n_:4d}  ->  구간 폭 {2 * 1.96 * SIGMA / np.sqrt(n_):.2f}')

## 4. 여론조사의 ±3.1%p 직접 계산

비율 조사의 표준오차는 √(p(1−p)/n). 가장 불리한 p=0.5 로 두고
n 별 오차범위(= 1.96 × SE)를 계산합니다.


In [ ]:
for n_poll in [250, 500, 1000, 2000, 4000]:
    moe = 1.96 * np.sqrt(0.5 * 0.5 / n_poll)
    print(f'표본 {n_poll:5d}명  ->  오차범위 ±{moe * 100:.1f}%p')

표본 1,000명이 바로 뉴스의 **±3.1%p** 입니다. 4,000명으로 4배 늘려야 ±1.5%p, √n 의 저주죠.

## 정리

- **모집단/모수**(알고 싶은 전체/참값) vs **표본/통계량**(조사한 일부/잰 값).
- **95% 신뢰구간** = x̄ ± 1.96·σ/√n. 1.96(6장 정규분포) × 표준오차(7장)의 조립품.
- "95%" 는 **방법의 적중률**: 100번 중 약 95번 참값을 포함 (개별 구간의 확률이 아님!).
- 🤖 **AI 연결**: "정확도 87.3%" 도 테스트셋(표본)에서 잰 통계량이라, 오차범위 없는 소수점 비교는 무의미할 수 있습니다.

다음 노트북 `ch09_hypothesis_test.ipynb`: "그 차이, 우연 아니야?"를 판정합니다.
